# 作业 03：在 PyTorch 中调用交叉熵

本次只判断一件事：**有没有行人？** 不使用图片、CNN 或数据下载。我们用一个简单函数产生两个预测概率，把注意力放在最后的损失函数调用上。

类别的顺序始终固定：

| 类别编号 | 含义 | one-hot 标签 y |
|---|---|---|
| 0 | 没有行人 | `[1, 0]` |
| 1 | 有行人 | `[0, 1]` |

`y` 和 `y_pred` 都表示**同一个样本的两个类别**，shape 为 `(2,)`。`y_pred[0]` 是没有行人的概率，`y_pred[1]` 是有行人的概率，两者相加为 1。

目标：会调用 `F.cross_entropy`，并能解释概率变大或变小时，损失为什么变化。


In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')


## 1. 一个输出两个概率的预测函数

为了便于观察，直接把输入 x 当作“有行人的预测概率”，于是“没有行人的概率”就是 `1-x`。这是人为设置的教学函数，不是训练好的行人识别器。

先取 x=0.2，则预测应为 `[0.8, 0.2]`。


In [2]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.2)
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())


x: 0.20000000298023224, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32


## 2. 最后调用 cross_entropy 计算损失

**PyTorch 的 `F.cross_entropy` 接收原始分数 logits，而不是直接接收概率。**

本例的函数已经输出概率，因此先用 `torch.log(y_pred)` 构造一组等价分数：对这组分数做 softmax，会还原原来的概率。这里两个概率均大于 0。

真实标签使用类别编号：`y=[1,0]` 对应编号 `0`，`y=[0,1]` 对应编号 `1`。用 `y.argmax()` 取出这个编号，得到整数标量。这里取的是**真实标签**的编号，不是预测的 argmax。

请重点看最后一行：损失由 `F.cross_entropy(...)` 计算。


In [3]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)


scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 0.2231435477733612, shape=(), dtype=torch.float32


### 对照手算

展开公式：

$$L=-[y_0\ln(p_0)+y_1\ln(p_1)].$$

真实标签为 `[1,0]` 时，只留下 $-\ln(p_0)$。当前 $p_0=0.8$，所以损失约为 **0.2231**。

下面仅用于核对；实际调用损失函数时，仍使用上面的 `F.cross_entropy`。


In [4]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))


手算：真实类别概率的负对数: 0.2231435328722, shape=(), dtype=torch.float32
与 cross_entropy 一致： True


## 3. 几个带数字的例子

下面固定真实标签为 `[1,0]`，即“没有行人”。只改变模型的预测概率。

| 真实标签 y | 预测概率 y_pred | 交叉熵 |
|---|---|---|
| `[1,0]` | `[0.9,0.1]` | $-\ln(0.9)\approx0.1054$ |
| `[1,0]` | `[0.5,0.5]` | $-\ln(0.5)\approx0.6931$ |
| `[1,0]` | `[0.1,0.9]` | $-\ln(0.1)\approx2.3026$ |
| `[1,0]` | `[0,1]` | $-\ln(0)\to+\infty$：把真实类别的概率设为 0 |
| `[1,0]` | `[1,0]` | $-\ln(1)=0$：完全相信真实类别 |

概率越接近真实标签，损失越小。概率精确为 0 的情况是边界示例；PyTorch 会在下面的错误预测例子中打印 `inf`，表示正无穷。


In [5]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()


y=[1.0, 0.0], y_pred=[0.8999999761581421, 0.10000000149011612]
真实类别概率=0.9000, loss=0.1054

y=[1.0, 0.0], y_pred=[0.5, 0.5]
真实类别概率=0.5000, loss=0.6931

y=[1.0, 0.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.1000, loss=2.3026

y=[1.0, 0.0], y_pred=[0.0, 1.0]
真实类别概率=0.0000, loss=inf

y=[1.0, 0.0], y_pred=[1.0, 0.0]
真实类别概率=1.0000, loss=0.0000



## 4. 换成“有行人”试试看

现在 y 为 `[0,1]`，真实类别是 1，应当关注第二个概率。下面预测为 `[0.1,0.9]`，损失约为 **0.1054**。同一个预测在上节被惩罚，在这里得到较小损失，是因为真实答案变了。


In [6]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)


y=[0.0, 1.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.9000, loss=0.1054


tensor(0.1054)

## 5. 真正的网络输出原始分数时，直接调用

前面的 `torch.log` 是为了配合本题“给定概率”的数值演示。真实网络通常直接输出两个原始分数，此时**直接把分数交给 cross_entropy**，不需要先 softmax 或取 log。

下面的 scores 为 `[2,-1]`，它们不是概率，也不需要加起来等于 1。softmax 只用来把概率打印给我们看，不是交叉熵的输入。


In [7]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)


原始分数 scores: tensor([ 2., -1.]), shape=(2,), dtype=torch.float32
给人看的预测概率: tensor([0.9526, 0.0474]), shape=(2,), dtype=torch.float32
直接调用 cross_entropy 得到的 loss: 0.04858732968568802, shape=(), dtype=torch.float32


## 你的实验（50分）

1. 把第 1 节的 x 改成 `0.8`，从第 1 节重新运行到手算核对。保持真实标签为 `[1,0]`。记录 y_pred 和损失，解释为什么损失变大。
2. 在下面补全代码：真实标签是“有行人”，预测概率为 `[0.8,0.2]`。打印两个张量的 shape，再调用 cross_entropy，预期损失约为 `1.6094`。
3. 用自己的话解释：为什么 `[1,0]` 对 `[0,1]` 的损失是无穷大，而 `[1,0]` 对 `[1,0]` 的损失是 0？
4. `F.cross_entropy(y_pred, label)` 与 `F.cross_entropy(torch.log(y_pred), label)` 中，哪一个才是在计算这里指定的预测概率对应的交叉熵？原始网络输出 scores 时，又该怎样调用？


### 实验 1：把 x 改成 0.8，看损失怎么变

真实标签固定为 `[1,0]`（没有行人）。上面第 1 节取 `x=0.2` 时，`y_pred=[0.8,0.2]`，损失约为 0.2231；下面把 `x` 改成 `0.8`，重新走一遍第 1 节到手算核对的完整流程。

In [8]:
# 实验 1：把第 1 节的 x 改成 0.8，真实标签仍然保持 [1,0]（没有行人）
# 第 1 节的原始单元格保留 x=0.2，这里另起变量做对照，两组结果都能留在输出里
x_exp = torch.tensor(0.8)
y_exp = torch.tensor([1., 0.])
y_pred_exp = predict(x_exp)

show('x（改成 0.8）', x_exp)
show('y（真实标签仍为没有行人）', y_exp)
show('y_pred（两个预测概率）', y_pred_exp)
show('预测概率之和', y_pred_exp.sum())

# 按第 2 节的流程：先由概率构造等价分数，再由真实标签取类别编号
scores_exp = torch.log(y_pred_exp)
label_exp = y_exp.argmax()

show('scores（传给损失函数的分数）', scores_exp)
show('softmax(scores)（还原概率）', scores_exp.softmax(dim=0))
show('label（真实类别编号）', label_exp)

loss_exp = F.cross_entropy(scores_exp, label_exp)
show('loss', loss_exp)

# 手算核对：真实标签是 [1,0]，公式里只留下 -ln(p_0)，这里 p_0 = 0.2
manual_exp = -torch.log(y_pred_exp[label_exp])
show('手算：真实类别概率的负对数 -ln(0.2)', manual_exp)
print('与 cross_entropy 一致：', torch.allclose(loss_exp, manual_exp))
print('对照 x=0.2 时的损失 0.2231，现在损失变成', round(loss_exp.item(), 4))


x（改成 0.8）: 0.800000011920929, shape=(), dtype=torch.float32
y（真实标签仍为没有行人）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32
scores（传给损失函数的分数）: tensor([-1.6094, -0.2231]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 1.6094379425048828, shape=(), dtype=torch.float32
手算：真实类别概率的负对数 -ln(0.2): 1.6094379425048828, shape=(), dtype=torch.float32
与 cross_entropy 一致： True
对照 x=0.2 时的损失 0.2231，现在损失变成 1.6094


In [9]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.8, 0.2])

# 打印 y 和 y_pred 的 shape
show('y（真实标签：有行人）', y)
show('y_pred（两个预测概率）', y_pred)

# 由真实 y 得到类别编号：[0,1] 表示真实类别是 1，dtype 为 torch.int64
label = y.argmax()
show('label（真实类别编号）', label)

# 由 y_pred 构造等价 scores，再调用 F.cross_entropy
scores = torch.log(y_pred)
show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))

loss = F.cross_entropy(scores, label)
show('loss', loss)

# 核对：真实类别是 1，其概率为 0.2，损失应为 -ln(0.2) ≈ 1.6094
print('手算 -ln(0.2) =', round(-torch.log(torch.tensor(0.2)).item(), 4),
      '，与 cross_entropy 一致：', torch.allclose(loss, -torch.log(y_pred[label])))


y（真实标签：有行人）: tensor([0., 1.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 1, shape=(), dtype=torch.int64
scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
loss: 1.6094379425048828, shape=(), dtype=torch.float32
手算 -ln(0.2) = 1.6094 ，与 cross_entropy 一致： True


### 我的回答 (50分)

- **修改 x 后的预测与损失**：把第 1 节的 x 从 0.2 改成 0.8（真实标签仍是 `[1,0]`，即没有行人），`predict` 返回 y_pred = `[0.2, 0.8]`（没有行人 0.2、有行人 0.8），损失 loss = **1.6094**，手算 -ln(0.2) = 1.6094 完全一致。为什么损失变大：真实类别是 0（没有行人），交叉熵只看真实类别那个概率 p_0，loss = -ln(p_0)。x=0.2 时 p_0=0.8，-ln(0.8)=0.2231；x=0.8 时 p_0 掉到 0.2，-ln(0.2)=1.6094。模型本来给"没有行人"0.8 的把握，改完只剩 0.2，等于越来越不相信正确答案；-ln 是减函数、概率越小值越大，所以损失从 0.2231 涨到 1.6094。

- **"有行人"实验的结果**：真实标签 y=`[0,1]`（有行人，类别编号 label=1），预测 y_pred=`[0.8,0.2]`，两个张量 shape 都是 `(2,)`。由 `y.argmax()` 得 label=1，用 `torch.log(y_pred)` 构造等价 scores=`[-0.2231,-1.6094]`，softmax 还原回 `[0.8,0.2]`。`F.cross_entropy(scores, label)` = **1.6094**，和预期一致（手算 -ln(0.2)=1.6094）。这里真实类别是 1，只看 p_1=0.2，模型给"有行人"的概率只有 0.2，所以损失大；这跟上一节同样的预测 `[0.8,0.2]` 在真实标签为 `[1,0]` 时损失小，是因为真实答案变了、关注的概率位置也跟着变。

- **两个极端例子的解释**：损失公式是 -ln(真实类别的预测概率)。
  - `[1,0]` 对 `[0,1]`：真实类别是 0，但模型给类别 0 的概率是 0，loss = -ln(0)。ln(0) 是负无穷，取负号后是正无穷，所以 PyTorch 打印 `inf`（此时 scores=log([0,1])=`[-inf, 0]`）。含义是模型 100% 断定"有行人"、把"没有行人"的概率压到 0，而真相恰恰是"没有行人"，这是最离谱的错误，惩罚无穷大。
  - `[1,0]` 对 `[1,0]`：真实类别是 0，模型给类别 0 的概率正好是 1，loss = -ln(1) = 0（此时 scores=log([1,0])=`[0, -inf]`）。含义是模型完全相信正确答案、毫无犹豫，预测和真相完全一致，所以零损失。
  - 两者对比说明：概率越靠近真实类别（越接近 1），损失越小越接近 0；越远离真实类别（越接近 0），损失越大直到无穷。

- **概率与原始分数应该怎样传入损失函数**：
  - 本题给的是**概率** y_pred（两个数相加为 1）。要"计算这里指定的预测概率对应的交叉熵"，必须传 **`F.cross_entropy(torch.log(y_pred), label)`**，即先取 log 变成等价分数再传。因为 `F.cross_entropy` 内部会先对输入做 softmax，而 softmax(log(p)) 正好还原成 p，得到的才是我们要的 -ln(p)。实测这样算出来是 1.6094，和手算一致。
  - 如果直接传 **`F.cross_entropy(y_pred, label)`** 是错的：它会把概率 `[0.8,0.2]` 当成原始分数再做一次 softmax，得到 `[0.6457,0.3543]`（已经不是原来的概率了），最终损失 1.0375，跟真实的 -ln(0.2)=1.6094 对不上。
  - 真实网络通常**直接输出原始分数 scores**（logits，不是概率、也不需要加起来等于 1，比如第 5 节的 `[2,-1]`）。这时**直接把 scores 交给 `F.cross_entropy(scores, label)`** 就行，不要自己先 softmax 或取 log——交叉熵内部会处理。softmax 只是我们想把分数看成概率、打印出来给人看时才用，不是损失函数的输入。

### 提交

补全实验代码和文字回答，重启内核后运行全部单元格并保存输出。示例中的 `inf` 是刻意展示的无穷大，不是代码异常。本题不要求编写训练循环。

参考：[PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。